# Question- Scrape movie review from IMDb website and clean the data includes revmoving dublicates,handling missing values,and standardizing format to ensure accurate and relabile analysis?

# Step 1: Import Dependencies

In [2]:
# Import the libraries
import requests                  # For making HTTP requests
from bs4 import BeautifulSoup     # For parsing HTML content
import pandas as pd           # For creating, storing and manipulating DataFrames
import json

# Step 2: Send a Request to IMDb Review

In [4]:
# Define the URL of the IMDb review
url = "https://www.imdb.com/list/ls095374765/"

In [1]:
#set the headers to mimic a browser request
headers={
   "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    
}

In [6]:
# Send an HTTP GET request to fetch the page content with headers
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    print("Successfully fetched the page!")
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

Successfully fetched the page!


In [7]:
# Send an HTTP GET request with the headers
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    print("Successfully fetched the page!")
else:
    print(f"Failed to retrieve the page. Status code: {response.status_code}")

# Display the response object (if needed)
response

Successfully fetched the page!


<Response [200]>

# Step 3:  Parse the HTML Content

In [9]:
#Parse the HTML content of the page using Beautiful Soup
soup = BeautifulSoup(response.content, 'html.parser')

In [10]:
reviews = []
for review in soup.find_all("div", class_="text show-more__control"):
    reviews.append(review.text.strip())

# Extract Movie Data

In [11]:
df = pd.DataFrame(reviews, columns=["Review"])
print(df.head())

Empty DataFrame
Columns: [Review]
Index: []


# Debugging

In [13]:
# Debugging: Print parse HTML to verify
print(soup.prettify()[:2000])  # View the full HTML structure

<!DOCTYPE html>
<html lang="en-US" xmlns:fb="http://www.facebook.com/2008/fbml" xmlns:og="http://opengraphprotocol.org/schema/">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width" name="viewport"/>
  <script>
   if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }
  </script>
  <script>
   window.addEventListener('load', (event) => {
        if (typeof window.csa !== 'undefined' && typeof window.csa === 'function') {
            var csaLatencyPlugin = window.csa('Content', {
                element: {
                    slotId: 'LoadTitle',
                    type: 'service-call'
                }
            });
            csaLatencyPlugin('mark', 'clickToBodyBegin', 1742263284205);
        }
    })
  </script>
  <title>
   Movie Reviews
  </title>
  <meta content="" data-id="main" name="description"/>
  <meta content="max-image-preview:large" name="robots"/>
  <script type="application/ld+json">
   {"@type":"ItemList","itemListElement":[{"@type":"L

# Step 4: Extract the movie details

In [15]:
# Find the JSON-LD script tag
json_data = soup.find('script', type='application/ld+json')

if json_data:    # Parse the JSON data
    data = json.loads(json_data.string) 

In [16]:
# Extract titles and ratings
titles = []
descriptions = []
ratings = []
durations = []
stars = []

In [17]:
# Check if the data contains the expected structure
if 'itemListElement' in data:
    for item in data['itemListElement']:
        movie = item['item']
        
# Extract movie details
        titles.append(movie['name'])  # Movie name
        descriptions.append(movie['description'])  # Movie description
        ratings.append(float(movie['aggregateRating']['ratingValue'])) # Rating value
        durations.append(movie['duration']) # Movie duration

# Step : Create a DataFrame

In [18]:
# Create a DataFrame to store the scraped data
df = pd.DataFrame({
    'Title': titles,
    'Description': descriptions,
    'Rating': ratings,
    'Duration': durations
 })

In [19]:
df.head()

,Title,Description,Rating,Duration
0,Aladdin,"Aladdin, a kind thief, woos Jasmine, the princ...",6.9,PT2H8M
1,It Chapter Two,Twenty-seven years after their first encounter...,6.5,PT2H49M
2,Joker,"Arthur Fleck, a party clown and a failed stand...",8.3,PT2H2M
3,Dolemite Is My Name,Eddie Murphy portrays real-life legend Rudy Ra...,7.2,PT1H58M
4,Anna,Beneath Anna Poliatova&apos;s striking beauty ...,6.7,PT1H58M


# Data Filtering and Cleaning

In [39]:
# Step 1: Display summary information about the DataFrame
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        250 non-null    object 
 1   Description  250 non-null    object 
 2   Rating       250 non-null    float64
 3   Duration     250 non-null    object 
dtypes: float64(1), object(3)
memory usage: 7.9+ KB
None


In [41]:
# Step 2: Check for null values
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
Title          0
Description    0
Rating         0
Duration       0
dtype: int64


In [49]:
# Step 3 : Handle Missing rating with median value (if needed)
df['Rating'].fillna(df['Rating'].median(), inplace=True)

# Check for duplicates and remove them
print("\nDuplicate Rows Before:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Duplicate Rows After:", df.duplicated().sum())


Duplicate Rows Before: 0
Duplicate Rows After: 0


C:\Users\kishr\AppData\Local\Temp\ipykernel_4752\3208306335.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Rating'].fillna(df['Rating'].median(), inplace=True)


In [55]:
# Display some sample data
print("\nSample Data:")
print(df.sample(5))


Sample Data:
                                      Title  \
110                              Uncle Buck   
40   The Conjuring: The Devil Made Me Do It   
242                       The Toxic Avenger   
231                Deadpool &amp; Wolverine   
200                          The Big Kahuna   

                                           Description  Rating Duration  
110  Laid back commitment-phobe Buck babysits his b...     7.1  PT1H40M  
40   Arne Cheyenne Johnson stabs and murders his la...     6.3  PT1H52M  
242  Tromaville has a monstrous new hero. The Toxic...     6.2  PT1H22M  
231  Deadpool is offered a place in the Marvel Cine...     7.6   PT2H8M  
200  Two veteran salesmen dissect a sales pitch to ...     6.5  PT1H30M  


In [61]:
# Filter: Show only movies with a rating above 8
high_rated_movies = df[df['Rating'] > 8]
print("\nHighly Rated Movies:")
print(high_rated_movies)


Highly Rated Movies:
                                                 Title  \
2                                                Joker   
68                                 Catch Me If You Can   
161                                       Living Proof   
182  Paradise Lost: The Child Murders at Robin Hood...   
186   Dear Zachary: A Letter to a Son About His Father   
217                                     V for Vendetta   

                                           Description  Rating Duration  
2    Arthur Fleck, a party clown and a failed stand...     8.3   PT2H2M  
68   Barely 17 yet, Frank is a skilled forger who h...     8.1  PT2H21M  
161  When a young man is diagnosed with a debilitat...     8.1  PT1H33M  
182  A horrific triple child murder leads to an ind...     8.2  PT2H30M  
186  A filmmaker decides to memorialize a murdered ...     8.5  PT1H35M  
217  In a future British dystopian society, a shado...     8.1  PT2H12M  


In [63]:
# Filter: Show only movies with a rating above 9
high_rated_movies = df[df['Rating'] > 9]
print("\nHighly Rated Movies:")
print(high_rated_movies)


Highly Rated Movies:
Empty DataFrame
Columns: [Title, Description, Rating, Duration]
Index: []


In [65]:
# Save the DataFrame to a CSV file
df.to_csv('topIMBDmovies.csv', index=False) # Setting index=False

In [67]:
df

,Title,Description,Rating,Duration
0,Aladdin,"Aladdin, a kind thief, woos Jasmine, the princ...",6.9,PT2H8M
1,It Chapter Two,Twenty-seven years after their first encounter...,6.5,PT2H49M
2,Joker,"Arthur Fleck, a party clown and a failed stand...",8.3,PT2H2M
3,Dolemite Is My Name,Eddie Murphy portrays real-life legend Rudy Ra...,7.2,PT1H58M
4,Anna,Beneath Anna Poliatova&apos;s striking beauty ...,6.7,PT1H58M
...,...,...,...,...
245,Cool World,When Jack Deebs was behind bars he found escap...,4.8,PT1H42M
246,Batman: The Killing Joke,"As Batman hunts for the escaped Joker, the Clo...",6.4,PT1H16M
247,Barb Wire,"During the Second American Civil War in 2017, ...",3.6,PT1H38M
248,Push,Two young Americans with special abilities mus...,6.1,PT1H51M
